In [1]:
!nvidia-smi

Fri Sep 25 07:13:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ortools
!pip install pycuda

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 29.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 w

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 11.7 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp313-cp313-linux_x86_64.whl size=5280313 sha256=46d770e64bb2a768b21fa24431a86ce339221d87afc329f96b2c6f440f16de26
  Stored in directory: /root/.cache/pip/wheels/ce/26/46/c519675fcb0e5e17bab8e85b6676528c40d12d794182340e85
Successfully built pycuda


In [3]:
from ortools.algorithms.python import knapsack_solver
import pycuda.autoinit
import pycuda.driver as cuda
import numpy as np
from pycuda.compiler import SourceModule
import time

In [4]:
# Colab starts with an empty filesystem, so pull the project in to get problems.py.
# Re-run this cell after a runtime restart: the clone survives, sys.path does not.
import os
import sys

REPO_URL = "https://github.com/andrewrowell/subset-sum-gpu.git"
REPO_DIR = "/content/subset-sum-gpu"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

Cloning into '/content/subset-sum-gpu'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 138 (delta 61), reused 101 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 233.82 KiB | 1.24 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [5]:
NUMBER_OF_TRIALS = 100
PREVIEW_ITEMS = 16

import numpy as np

import problems

# Every implementation in this project solves the identical set of problems by
# reading them from problems.py.
problem_set = problems.generate()

num_problems = problem_set.num_problems
num_items_per_problem = problem_set.num_items
max_capacity = problem_set.max_capacity
capacities = problem_set.capacities

items = problem_set.items

print(f"{num_problems} problems, {num_items_per_problem} items each, capacity at most {max_capacity}")
for i in range(3):
    row = problem_set.items[i]
    head = ", ".join(str(item) for item in row[:PREVIEW_ITEMS])
    rest = f", ... ({len(row) - PREVIEW_ITEMS} more)" if len(row) > PREVIEW_ITEMS else ""
    print(f"Problem {i + 1}: capacity {problem_set.capacities[i]}, items [{head}{rest}]")

10000 problems, 16 items each, capacity at most 2000
Problem 1: capacity 1954, items [90, 774, 654, 439, 433, 858, 86, 697, 202, 95, 526, 975, 736, 761, 717, 786]
Problem 2: capacity 1296, items [513, 128, 839, 450, 500, 371, 183, 926, 781, 644, 403, 822, 545, 443, 451, 228]
Problem 3: capacity 62, items [93, 555, 888, 64, 858, 827, 277, 632, 166, 758, 700, 355, 68, 970, 446, 893]


In [6]:
# OR-Tools Section

# This section should run in Colab T4
import platform
print(platform.node())

# Function to solve a single subset sum problem using OR-Tools
def solve_subset_sum(items, capacity, problem_idx, results):
    solver = knapsack_solver.KnapsackSolver(
        knapsack_solver.KNAPSACK_DYNAMIC_PROGRAMMING_SOLVER, 'SubsetSumExample')

    # Subset sum is knapsack with an item's value equal to its weight, so
    # OR-Tools is given the same array for both.
    solver.init(items, [items], [capacity])

    max_value = solver.solve()
    results[problem_idx] = max_value
    #print(f"Problem {problem_idx + 1}: Maximum value = {max_value}")

# Storage for results
results = [0] * num_problems

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    # Solve them one at a time. Threads only slow this down: OR-Tools holds the GIL
    # for the whole solve, so 10,000 of them just add scheduling overhead.
    for i in range(num_problems):
        solve_subset_sum(items[i], capacities[i], i, results)

    # Print execution time
    end_time = time.time()
    #print(f"Threaded execution time: {end_time - start_time:.6f} seconds")
    execution_times.append(end_time - start_time)

print(f"Average CPU execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Print the final results
#for i in range(num_problems):
for i in range(10):
    print(f"Final Result for Problem {i + 1}: Maximum value = {results[i]}")

0881d3a12815
Average CPU execution time: 0.321509 seconds
Final Result for Problem 1: Maximum value = 1954
Final Result for Problem 2: Maximum value = 1296
Final Result for Problem 3: Maximum value = 0
Final Result for Problem 4: Maximum value = 239
Final Result for Problem 5: Maximum value = 1349
Final Result for Problem 6: Maximum value = 1303
Final Result for Problem 7: Maximum value = 725
Final Result for Problem 8: Maximum value = 1291
Final Result for Problem 9: Maximum value = 717
Final Result for Problem 10: Maximum value = 939


In [7]:
# Bitset DP Section
#
# The same dynamic program the GPU kernels run, on the CPU. problems.best_total packs
# the reachable totals into the bits of a single integer, so one shift and one OR
# advance the whole table by one item.
print(platform.node())

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    bitset_results = [
        problems.best_total(items[i], int(capacities[i])) for i in range(num_problems)
    ]

    # Print execution time
    end_time = time.time()
    execution_times.append(end_time - start_time)

print(f"Average bitset DP execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Check against the OR-Tools section above
mismatches = np.flatnonzero(np.array(bitset_results) != np.array(results))
print(f"{len(mismatches)} of {num_problems} results differ from OR-Tools")

# Print the results
#for i in range(num_problems):
for i in range(10):
    print(f"Problem {i + 1}: Maximum value = {bitset_results[i]}")

0881d3a12815
Average bitset DP execution time: 0.082854 seconds
0 of 10000 results differ from OR-Tools
Problem 1: Maximum value = 1954
Problem 2: Maximum value = 1296
Problem 3: Maximum value = 0
Problem 4: Maximum value = 239
Problem 5: Maximum value = 1349
Problem 6: Maximum value = 1303
Problem 7: Maximum value = 725
Problem 8: Maximum value = 1291
Problem 9: Maximum value = 717
Problem 10: Maximum value = 939


In [8]:
# CUDA kernel to solve multiple subset sum problems. One block solves one problem;
# its threads split the capacities 0..max_capacity between them.
#
# reachable[w] is 1 when some subset of the items seen so far sums to exactly w.
# Adding an item makes w reachable if w - item already was. Each item reads one
# table and writes the other, so no thread can see a value that already includes
# the current item, which would let that item be used twice.
kernel_code = """
__global__ void subset_sum(const int *items, const int *capacities, int *max_values,
                           int num_items, int max_capacity) {
    int width = max_capacity + 1;
    extern __shared__ unsigned char tables[];
    unsigned char *current = tables;
    unsigned char *next = tables + width;

    // Only the empty subset exists before any item is considered
    for (int w = threadIdx.x; w < width; w += blockDim.x) {
        current[w] = (w == 0);
    }
    __syncthreads();

    const int *problem_items = items + blockIdx.x * num_items;
    for (int i = 0; i < num_items; i++) {
        int item = problem_items[i];
        for (int w = threadIdx.x; w < width; w += blockDim.x) {
            next[w] = current[w] | (w >= item ? current[w - item] : 0);
        }
        __syncthreads();

        unsigned char *swap = current;
        current = next;
        next = swap;
    }

    // The answer is the largest reachable sum that fits in the capacity
    if (threadIdx.x == 0) {
        int w = capacities[blockIdx.x];
        while (!current[w]) {
            w--;
        }
        max_values[blockIdx.x] = w;
    }
}
"""

# Compile the kernel code
mod = SourceModule(kernel_code)
subset_sum = mod.get_function("subset_sum")

# Allocate and fill the device buffers once, outside the timed loop. Unlike Apple
# silicon, this really is a transfer across PCIe to separate device memory, so it is
# worth doing once rather than on every trial.
items_gpu = cuda.mem_alloc(items.nbytes)
capacities_gpu = cuda.mem_alloc(capacities.nbytes)
max_values_gpu = cuda.mem_alloc(num_problems * 4)
cuda.memcpy_htod(items_gpu, items)
cuda.memcpy_htod(capacities_gpu, capacities)

max_values = np.zeros(num_problems, dtype=np.int32)

# Two tables of max_capacity + 1 bytes each
shared_memory_size = 2 * (max_capacity + 1)
threads_per_block = min(max_capacity + 1, 1024)

execution_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    # Launch the kernel
    subset_sum(items_gpu, capacities_gpu, max_values_gpu,
               np.int32(num_items_per_problem), np.int32(max_capacity),
               block=(threads_per_block, 1, 1), grid=(num_problems, 1),
               shared=shared_memory_size)

    # Copy the result back to the CPU; this synchronizes with the kernel
    cuda.memcpy_dtoh(max_values, max_values_gpu)

    # Print execution time
    end_time = time.time()
    execution_times.append(end_time - start_time)

print(f"Average PyCUDA execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Check against the OR-Tools section above
mismatches = np.flatnonzero(max_values != np.array(results))
print(f"{len(mismatches)} of {num_problems} results differ from OR-Tools")

# Print the results
#for i in range(num_problems):
for i in range(10):
    print(f"Problem {i + 1}: Maximum value = {max_values[i]}")

Average PyCUDA execution time: 0.003320 seconds
0 of 10000 results differ from OR-Tools
Problem 1: Maximum value = 1954
Problem 2: Maximum value = 1296
Problem 3: Maximum value = 0
Problem 4: Maximum value = 239
Problem 5: Maximum value = 1349
Problem 6: Maximum value = 1303
Problem 7: Maximum value = 725
Problem 8: Maximum value = 1291
Problem 9: Maximum value = 717
Problem 10: Maximum value = 939
